
# Notebook 22 — Cross-Family Transfer and OOD Universality Tests

This notebook evaluates whether residual universality structure transfers to
previously unseen graph families.

Core question:

> Do residual manifold embeddings generalize beyond graph topology families used
> to construct an original universality manifold?

This version includes fixes for:
- `NaN` values from disconnected graph path-length calculations,
- deprecated `np.sum(generator)` entropy calculation,
- robust feature imputation before PCA,
- robust repository root detection in Colab,
- a notebook output zip and optional Colab download block.


In [ ]:

import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

np.random.seed(42)

def detect_repo_root():
    cwd = Path.cwd()
    if cwd.name == "notebooks":
        return cwd.parent
    if (cwd / "notebooks").exists() or (cwd / ".git").exists():
        return cwd
    if Path("/content").exists():
        candidates = list(Path("/content").glob("*"))
        for c in candidates:
            if c.is_dir() and ((c / "notebooks").exists() or (c / ".git").exists()):
                return c
        return Path("/content")
    return cwd

ROOT = detect_repo_root()
RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"
DOCS = ROOT / "docs"

RESULTS.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)
DOCS.mkdir(exist_ok=True)

print("cwd:", Path.cwd())
print("repo root:", ROOT)
print("results:", RESULTS)
print("figures:", FIGURES)
print("docs:", DOCS)


## 1. Graph family generators

In [ ]:

def safe_connected_largest_component(G):
    # Convert possible multigraphs to simple undirected graphs.
    G = nx.Graph(G)
    G.remove_edges_from(nx.selfloop_edges(G))

    if len(G) == 0:
        return G

    if nx.is_connected(G):
        return G

    largest = max(nx.connected_components(G), key=len)
    return G.subgraph(largest).copy()

def ring_lattice_graph(n):
    return nx.watts_strogatz_graph(n, 4, 0.0, seed=100 + n)

def small_world_graph(n):
    return nx.watts_strogatz_graph(n, 4, 0.25, seed=200 + n)

def er_graph(n):
    G = nx.erdos_renyi_graph(n, 0.10, seed=300 + n)
    return safe_connected_largest_component(G)

def scale_free_graph(n):
    return nx.barabasi_albert_graph(n, 3, seed=400 + n)

def modular_clustered_graph(n):
    sizes = [n // 2, n - n // 2]
    probs = [[0.18, 0.012], [0.012, 0.18]]
    G = nx.stochastic_block_model(sizes, probs, seed=500 + n)
    return safe_connected_largest_component(G)

BASE_GENERATORS = {
    "ring lattice": ring_lattice_graph,
    "small world": small_world_graph,
    "Erdős–Rényi": er_graph,
    "scale free": scale_free_graph,
    "modular clustered": modular_clustered_graph,
}

# OOD graph families

def cycle_with_chords(n):
    G = nx.cycle_graph(n)
    for i in range(0, n, 4):
        G.add_edge(i, (i + n // 2) % n)
    return G

def rewired_ring(n):
    return nx.watts_strogatz_graph(n, 6, 0.45, seed=600 + n)

def degree_corrected_random(n):
    rng = np.random.default_rng(700 + n)
    degs = rng.poisson(4, n)
    degs = np.maximum(degs, 1)
    if degs.sum() % 2:
        degs[0] += 1
    G = nx.configuration_model(degs, seed=800 + n)
    return safe_connected_largest_component(G)

def hub_spoke_modular(n):
    half = max(4, n // 2)
    G1 = nx.barabasi_albert_graph(half, 2, seed=900 + n)
    G2 = nx.barabasi_albert_graph(n - half, 2, seed=1000 + n)
    G2 = nx.relabel_nodes(G2, {node: node + half for node in G2.nodes})
    G = nx.compose(G1, G2)
    G.add_edge(0, half)
    G.add_edge(1, half + 1 if half + 1 < n else half)
    return safe_connected_largest_component(G)

def two_block_bridge(n):
    sizes = [n // 2, n - n // 2]
    probs = [[0.14, 0.004], [0.004, 0.14]]
    G = nx.stochastic_block_model(sizes, probs, seed=1100 + n)
    G = safe_connected_largest_component(G)
    return G

OOD_GENERATORS = {
    "cycle with chords": cycle_with_chords,
    "rewired ring": rewired_ring,
    "degree corrected random": degree_corrected_random,
    "hub spoke modular": hub_spoke_modular,
    "two block bridge": two_block_bridge,
}

GRAPH_SIZES = [16, 32, 64, 128]


## 2. Residual feature extraction

In [ ]:

def normalized_entropy(weights):
    weights = np.asarray(weights, dtype=float)
    weights = np.abs(weights)
    total = weights.sum()
    if total <= 0:
        return 0.0
    p = weights / total
    p = p[p > 0]
    return float(-sum(float(pi) * np.log(float(pi)) for pi in p) / np.log(len(weights)))

def safe_average_path_length(G):
    G = safe_connected_largest_component(G)
    if len(G) <= 1:
        return 0.0
    try:
        return float(nx.average_shortest_path_length(G))
    except Exception:
        return 0.0

def graph_features(G):
    G = safe_connected_largest_component(G)

    n_nodes = max(len(G.nodes), 1)
    n_edges = len(G.edges)

    A = nx.to_numpy_array(G)
    eigvals = np.linalg.eigvalsh(A) if A.size else np.array([0.0])
    eigvals = np.sort(np.abs(eigvals))[::-1]

    degree = np.array([d for _, d in G.degree()], dtype=float)
    if len(degree) == 0:
        degree = np.array([0.0])

    clustering_values = list(nx.clustering(G).values()) if len(G) else [0.0]
    clustering = float(np.mean(clustering_values)) if clustering_values else 0.0

    path_length = safe_average_path_length(G)

    residual = eigvals - np.mean(eigvals)
    abs_residual = np.abs(residual)

    total_abs = float(abs_residual.sum())
    mean_abs = float(abs_residual.mean()) if len(abs_residual) else 0.0
    max_abs = float(abs_residual.max()) if len(abs_residual) else 0.0

    curvature = float(np.sum(np.abs(np.diff(residual)))) if len(residual) > 1 else 0.0
    localization = max_abs / (mean_abs + 1e-9)

    spectral_gap = float(eigvals[0] - eigvals[1]) if len(eigvals) > 1 else float(eigvals[0])
    spectral_ratio = float(eigvals[1] / (eigvals[0] + 1e-9)) if len(eigvals) > 1 else 0.0

    density = float(2 * n_edges / max(n_nodes * (n_nodes - 1), 1))

    return {
        "mean_abs_residual": mean_abs,
        "max_abs_residual": max_abs,
        "total_residual_energy": float(np.sum(residual ** 2)),
        "residual_localization": float(localization),
        "residual_entropy": normalized_entropy(abs_residual),
        "residual_curvature": curvature,
        "mean_degree": float(np.mean(degree)),
        "degree_std": float(np.std(degree)),
        "clustering": clustering,
        "path_length": path_length,
        "spectral_gap": spectral_gap,
        "spectral_ratio": spectral_ratio,
        "density": density,
        "n_connected_nodes": int(n_nodes),
        "n_edges": int(n_edges),
    }


## 3. Build training manifold

In [ ]:

rows = []

for topology, generator in BASE_GENERATORS.items():
    for N in GRAPH_SIZES:
        G = generator(N)
        feats = graph_features(G)
        feats["topology"] = topology
        feats["N"] = N
        feats["source"] = "train"
        rows.append(feats)

train_df = pd.DataFrame(rows)

feature_cols = [
    c for c in train_df.columns
    if c not in ["topology", "N", "source"]
]

# Clean feature table before PCA.
train_feature_df = train_df[feature_cols].replace([np.inf, -np.inf], np.nan)

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()
pca = PCA(n_components=2)

X_train_imputed = imputer.fit_transform(train_feature_df)
X_train_scaled = scaler.fit_transform(X_train_imputed)
train_pca = pca.fit_transform(X_train_scaled)

train_df["PC1"] = train_pca[:, 0]
train_df["PC2"] = train_pca[:, 1]

train_df.to_csv(RESULTS / "ood_train_residual_features.csv", index=False)

print("feature columns:", feature_cols)
print("PCA variance:", pca.explained_variance_ratio_)
train_df.head()


## 4. Project OOD families into learned manifold

In [ ]:

ood_rows = []

for topology, generator in OOD_GENERATORS.items():
    for N in GRAPH_SIZES:
        G = generator(N)
        feats = graph_features(G)
        feats["topology"] = topology
        feats["N"] = N
        feats["source"] = "ood"
        ood_rows.append(feats)

ood_df = pd.DataFrame(ood_rows)

ood_feature_df = ood_df[feature_cols].replace([np.inf, -np.inf], np.nan)
X_ood_imputed = imputer.transform(ood_feature_df)
X_ood_scaled = scaler.transform(X_ood_imputed)
ood_pca = pca.transform(X_ood_scaled)

ood_df["PC1"] = ood_pca[:, 0]
ood_df["PC2"] = ood_pca[:, 1]

ood_df.to_csv(RESULTS / "ood_residual_features.csv", index=False)

ood_df.head()


## 5. OOD transfer embedding

In [ ]:

plt.figure(figsize=(11, 8))

for topology in train_df["topology"].unique():
    sub = train_df[train_df["topology"] == topology].sort_values("N")
    plt.plot(sub["PC1"], sub["PC2"], marker="o", linewidth=2, alpha=0.65, label=topology)

for topology in ood_df["topology"].unique():
    sub = ood_df[ood_df["topology"] == topology].sort_values("N")
    plt.plot(sub["PC1"], sub["PC2"], marker="x", markersize=9, linewidth=2.2, linestyle="--", label=f"OOD: {topology}")
    for _, row in sub.iterrows():
        plt.text(row["PC1"], row["PC2"], f"N={int(row['N'])}", fontsize=8)

plt.axhline(0, linestyle="--", linewidth=0.8, color="black", alpha=0.45)
plt.axvline(0, linestyle="--", linewidth=0.8, color="black", alpha=0.45)

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}%)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}%)")
plt.title("OOD transfer embedding")
plt.grid(alpha=0.3)
plt.legend(fontsize=8, loc="best")

path = FIGURES / "ood_transfer_embedding.png"
plt.savefig(path, dpi=220, bbox_inches="tight")
plt.show()

print("saved:", path)


## 6. Nearest-family transfer assignment

In [ ]:

centroids = train_df.groupby("topology")[["PC1", "PC2"]].mean()

assignment_rows = []

for _, row in ood_df.iterrows():
    vec = np.array([row["PC1"], row["PC2"]], dtype=float)

    distances = {
        topology: float(np.linalg.norm(vec - centroids.loc[topology].values))
        for topology in centroids.index
    }

    ordered = sorted(distances.items(), key=lambda item: item[1])
    nearest = ordered[0]
    second = ordered[1]

    assignment_rows.append({
        "ood_topology": row["topology"],
        "N": int(row["N"]),
        "nearest_family": nearest[0],
        "nearest_distance": nearest[1],
        "second_family": second[0],
        "second_distance": second[1],
        "confidence_gap": second[1] - nearest[1],
        "relative_confidence": (second[1] - nearest[1]) / max(second[1], 1e-9),
    })

assign_df = pd.DataFrame(assignment_rows)
assign_df.to_csv(RESULTS / "ood_nearest_family_assignment.csv", index=False)

assign_df.head()


## 7. Transfer confidence

In [ ]:

confidence_summary = (
    assign_df.groupby("ood_topology")[["confidence_gap", "relative_confidence"]]
    .mean()
    .sort_values("relative_confidence")
)

plt.figure(figsize=(9, 5))
plt.barh(confidence_summary.index, confidence_summary["relative_confidence"])
plt.xlabel("mean relative confidence")
plt.title("OOD transfer confidence")
plt.grid(alpha=0.3, axis="x")

path = FIGURES / "ood_transfer_confidence.png"
plt.savefig(path, dpi=220, bbox_inches="tight")
plt.show()

print("saved:", path)
confidence_summary


## 8. OOD transfer similarity matrix

In [ ]:

similarity_rows = []

for topology in ood_df["topology"].unique():
    ood_centroid = ood_df[ood_df["topology"] == topology][["PC1", "PC2"]].mean().values

    row = {}
    for base_topology in centroids.index:
        base_centroid = centroids.loc[base_topology].values
        distance = np.linalg.norm(ood_centroid - base_centroid)
        row[base_topology] = float(np.exp(-distance / 3.0))

    similarity_rows.append(pd.Series(row, name=topology))

sim_df = pd.DataFrame(similarity_rows)
sim_df.to_csv(RESULTS / "ood_transfer_similarity_matrix.csv")

plt.figure(figsize=(9, 6))
im = plt.imshow(sim_df.values, aspect="auto")

plt.xticks(range(len(sim_df.columns)), sim_df.columns, rotation=45, ha="right")
plt.yticks(range(len(sim_df.index)), sim_df.index)

for i in range(sim_df.shape[0]):
    for j in range(sim_df.shape[1]):
        plt.text(j, i, f"{sim_df.iloc[i, j]:.2f}", ha="center", va="center")

plt.colorbar(im, label="transfer similarity")
plt.title("OOD transfer similarity matrix")
plt.tight_layout()

path = FIGURES / "ood_transfer_similarity_matrix.png"
plt.savefig(path, dpi=220, bbox_inches="tight")
plt.show()

print("saved:", path)
sim_df


## 9. OOD nearest-family map

In [ ]:

plt.figure(figsize=(11, 8))

for topology in train_df["topology"].unique():
    sub = train_df[train_df["topology"] == topology].sort_values("N")
    plt.scatter(sub["PC1"], sub["PC2"], alpha=0.30, s=120)
    c = centroids.loc[topology]
    plt.scatter(c["PC1"], c["PC2"], marker="*", s=260, color="black")
    plt.text(c["PC1"], c["PC2"], topology, fontsize=9, weight="bold")

for topology in ood_df["topology"].unique():
    sub = ood_df[ood_df["topology"] == topology].sort_values("N")
    plt.plot(sub["PC1"], sub["PC2"], marker="o", linewidth=2.2, label=topology)

    endpoint = sub.iloc[-1]
    nearest = assign_df[
        (assign_df["ood_topology"] == topology)
        & (assign_df["N"] == int(endpoint["N"]))
    ].iloc[0]["nearest_family"]

    c = centroids.loc[nearest]
    plt.plot(
        [endpoint["PC1"], c["PC1"]],
        [endpoint["PC2"], c["PC2"]],
        linestyle=":",
        color="black",
        alpha=0.65,
    )

plt.axhline(0, linestyle="--", linewidth=0.8, color="black", alpha=0.45)
plt.axvline(0, linestyle="--", linewidth=0.8, color="black", alpha=0.45)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("OOD nearest-family map")
plt.grid(alpha=0.3)
plt.legend(fontsize=8, loc="best")

path = FIGURES / "ood_nearest_family_map.png"
plt.savefig(path, dpi=220, bbox_inches="tight")
plt.show()

print("saved:", path)


## 10. Fixed-point transfer test

In [ ]:

fixed_points = (
    train_df.sort_values("N")
    .groupby("topology")[["PC1", "PC2"]]
    .last()
)

fp_rows = []

for topology in ood_df["topology"].unique():
    sub = ood_df[ood_df["topology"] == topology].sort_values("N")
    endpoint = sub.iloc[-1][["PC1", "PC2"]].values.astype(float)

    distances = {}
    for known_topology in fixed_points.index:
        fp = fixed_points.loc[known_topology].values.astype(float)
        distances[known_topology] = float(np.linalg.norm(endpoint - fp))

    nearest_fp = min(distances, key=distances.get)

    assigned_family = assign_df[
        (assign_df["ood_topology"] == topology)
        & (assign_df["N"] == int(sub.iloc[-1]["N"]))
    ].iloc[0]["nearest_family"]

    fp_rows.append({
        "ood_topology": topology,
        "endpoint_N": int(sub.iloc[-1]["N"]),
        "nearest_fixed_point": nearest_fp,
        "distance_to_nearest_fixed_point": distances[nearest_fp],
        "assigned_family": assigned_family,
        "distance_to_assigned_family_fixed_point": distances[assigned_family],
    })

fp_df = pd.DataFrame(fp_rows)
fp_df.to_csv(RESULTS / "ood_fixed_point_transfer.csv", index=False)

plt.figure(figsize=(9, 5))
plot_df = fp_df.sort_values("distance_to_nearest_fixed_point")
plt.barh(plot_df["ood_topology"], plot_df["distance_to_nearest_fixed_point"])
plt.xlabel("distance to nearest known endpoint")
plt.title("OOD fixed-point transfer distance")
plt.grid(alpha=0.3, axis="x")

path = FIGURES / "ood_fixed_point_transfer.png"
plt.savefig(path, dpi=220, bbox_inches="tight")
plt.show()

print("saved:", path)
fp_df


## 11. Transfer stability across graph size

In [ ]:

stability_rows = []

for topology in ood_df["topology"].unique():
    sub = assign_df[assign_df["ood_topology"] == topology].sort_values("N")

    nearest_sequence = list(sub["nearest_family"])
    switches = sum(
        nearest_sequence[i] != nearest_sequence[i - 1]
        for i in range(1, len(nearest_sequence))
    )

    stability_rows.append({
        "ood_topology": topology,
        "nearest_family_sequence": " → ".join(nearest_sequence),
        "n_family_switches": int(switches),
        "mean_relative_confidence": float(sub["relative_confidence"].mean()),
    })

stability_df = pd.DataFrame(stability_rows)
stability_df.to_csv(RESULTS / "ood_transfer_stability.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

plot_df = stability_df.sort_values("n_family_switches")
axes[0].barh(plot_df["ood_topology"], plot_df["n_family_switches"])
axes[0].set_title("nearest-family switches")
axes[0].set_xlabel("count")
axes[0].grid(alpha=0.3, axis="x")

plot_df = stability_df.sort_values("mean_relative_confidence")
axes[1].barh(plot_df["ood_topology"], plot_df["mean_relative_confidence"])
axes[1].set_title("mean relative confidence")
axes[1].set_xlabel("confidence")
axes[1].grid(alpha=0.3, axis="x")

plt.suptitle("OOD transfer stability across graph size")
plt.tight_layout()

path = FIGURES / "ood_transfer_stability.png"
plt.savefig(path, dpi=220, bbox_inches="tight")
plt.show()

print("saved:", path)
stability_df


## 12. Summary export

In [ ]:

summary = {
    "notebook": "22_cross_family_transfer_ood_universality_tests.ipynb",
    "core_question": "Do residual manifold embeddings generalize beyond training topology families?",
    "core_claim": (
        "OOD residual features partially transfer into known universality branches, "
        "with hybrid graph families occupying ambiguous or bridge-like regions."
    ),
    "train_topologies": list(BASE_GENERATORS.keys()),
    "ood_topologies": list(OOD_GENERATORS.keys()),
    "graph_sizes": GRAPH_SIZES,
    "feature_columns": feature_cols,
    "pca_variance_ratio": [float(x) for x in pca.explained_variance_ratio_],
    "mean_relative_confidence": float(assign_df["relative_confidence"].mean()),
    "mean_confidence_gap": float(assign_df["confidence_gap"].mean()),
    "most_confident_ood_family": str(confidence_summary["relative_confidence"].idxmax()),
    "least_confident_ood_family": str(confidence_summary["relative_confidence"].idxmin()),
    "figures": [
        "ood_transfer_embedding.png",
        "ood_transfer_confidence.png",
        "ood_transfer_similarity_matrix.png",
        "ood_nearest_family_map.png",
        "ood_fixed_point_transfer.png",
        "ood_transfer_stability.png",
    ],
    "results": [
        "ood_train_residual_features.csv",
        "ood_residual_features.csv",
        "ood_nearest_family_assignment.csv",
        "ood_transfer_similarity_matrix.csv",
        "ood_fixed_point_transfer.csv",
        "ood_transfer_stability.csv",
        "ood_transfer_summary.json",
    ],
}

summary_path = RESULTS / "ood_transfer_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

md_lines = [
    "# Notebook 22 — Cross-Family Transfer and OOD Universality Tests",
    "",
    "## Core question",
    "",
    "Do residual manifold embeddings generalize beyond training topology families?",
    "",
    "## Core claim",
    "",
    summary["core_claim"],
    "",
    "## OOD families",
    "",
]

for family in OOD_GENERATORS:
    md_lines.append(f"- {family}")

md_lines += [
    "",
    "## Key outputs",
    "",
    "- `figures/ood_transfer_embedding.png`",
    "- `figures/ood_transfer_similarity_matrix.png`",
    "- `figures/ood_nearest_family_map.png`",
    "- `figures/ood_fixed_point_transfer.png`",
    "- `figures/ood_transfer_stability.png`",
    "",
    "## Notes",
    "",
    "This notebook tests transfer behavior. It does not treat nearest-family assignment as proof of topology equivalence.",
]

md_path = DOCS / "notebook_22_ood_transfer.md"
md_path.write_text("\n".join(md_lines), encoding="utf-8")

print(json.dumps(summary, indent=2))
print("saved:", summary_path)
print("saved:", md_path)


## 13. Build notebook outputs archive

In [ ]:

zip_path = ROOT / "notebook_22_outputs.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in [FIGURES, RESULTS, DOCS]:
        if folder.exists():
            for file in folder.glob("*"):
                if file.is_file() and (
                    file.name.startswith("ood_")
                    or file.name.startswith("notebook_22")
                ):
                    zf.write(file, arcname=f"{folder.name}/{file.name}")

print("saved:", zip_path)

# Optional Colab download:
# from google.colab import files
# files.download(str(zip_path))



## Interpretation

Expected paper-facing interpretation:

- OOD graph families partially transfer into the learned residual manifold.
- Some graph families map cleanly to known branches.
- Hybrid systems can occupy ambiguous regions between learned families.
- Fixed-point transfer gives a compact way to compare OOD endpoints to known topology endpoints.
- Transfer confidence helps separate strong assignments from bridge-like assignments.

Recommended figures for a paper section:
1. `ood_transfer_embedding.png`
2. `ood_transfer_similarity_matrix.png`
3. `ood_nearest_family_map.png`
4. `ood_fixed_point_transfer.png`
5. `ood_transfer_stability.png`
